# How the Latest my_stock_picker Works

## Overview: Dual-Perspective Stock Analysis System

The latest my_stock_picker implementation uses **two separate finder agents** (Grok and Gemini) to get diverse stock perspectives, then consolidates research through a single researcher and rater.

## Architecture Diagram

```
┌─────────────────────────────────────────────────────────────────┐
│                    Manager (Gemini)                             │
│                   Coordinates Workflow                          │
└───────┬─────────────────────────────────────────────────────────┘
        │
        ├─────────────────────────────────────────────────────────┐
        │                                                         │
        ▼                                                         ▼
┌───────────────────────┐                         ┌───────────────────────┐
│  Finder Agent #1      │                         │  Finder Agent #2      │
│  (Grok-powered)       │                         │  (Gemini-powered)     │
│                       │                         │                       │
│  Focus: Fundamentals  │                         │  Focus: Momentum      │
│  Long-term trends     │                         │  Short-term catalysts │
└───────┬───────────────┘                         └───────┬───────────────┘
        │                                                 │
        │ Outputs 2-3 stocks                             │ Outputs 2-3 stocks
        │ (fundamental focus)                            │ (momentum focus)
        │                                                 │
        └─────────────────┬───────────────────────────────┘
                          │
                          ▼
                ┌─────────────────────┐
                │  Financial          │
                │  Researcher (Grok)  │
                │                     │
                │  Receives BOTH      │
                │  lists, researches  │
                │  all unique stocks  │
                └──────────┬──────────┘
                           │
                           │ Comprehensive research
                           │
                           ▼
                  ┌────────────────┐
                  │  Stock Rater   │
                  │  (Grok)        │
                  │                │
                  │  Rates 1-100   │
                  │  All stocks    │
                  └────────────────┘
```

## Agent Breakdown

### 1. Manager Agent (Gemini)
- **Role**: Coordinates the entire workflow
- **LLM**: Gemini 2.5 Flash (fast, efficient coordination)
- **Responsibilities**:
  - Delegates tasks to appropriate agents
  - Ensures workflow completion
  - In hierarchical mode, decides which agent works on which task

### 2. Finder Agent #1: Grok Perspective
- **Name**: `trending_company_finder_grok`
- **LLM**: Grok 4 Fast (thorough analysis)
- **Focus**: Fundamental analysis and long-term trends
- **Personality**: Methodical, thorough, seasoned market expert
- **Output**: 2-3 publicly traded companies with strong fundamentals
- **Output File**: `output/trending_companies_grok.json`

**What makes it unique:**
- Looks for companies with strong fundamentals
- Focuses on long-term value and trends
- Takes a more conservative, analytical approach

### 3. Finder Agent #2: Gemini Perspective
- **Name**: `trending_company_finder_gemini`
- **LLM**: Gemini 2.5 Flash (fast, momentum-focused)
- **Focus**: Momentum and short-term catalysts
- **Personality**: Quick-thinking, spots fast-moving trends
- **Output**: 2-3 publicly traded companies with strong momentum
- **Output File**: `output/trending_companies_gemini.json`
- **Special Instruction**: Try to find DIFFERENT companies than Grok finder

**What makes it unique:**
- Looks for momentum plays
- Focuses on short-term catalysts and news
- Takes a more aggressive, trend-following approach

### 4. Financial Researcher (Grok)
- **Name**: `financial_researcher`
- **LLM**: Grok 4 Fast (thorough research)
- **Input**: BOTH lists from the two finders
- **Responsibilities**:
  - Combine companies from both lists
  - Remove duplicates
  - Research each unique company thoroughly
- **Output**: Comprehensive research report
- **Output File**: `output/research_report.json`

### 5. Stock Rater (Grok)
- **Name**: `stock_rater`
- **LLM**: Grok 4 Fast (consistent rating)
- **Input**: Research from financial_researcher
- **Responsibilities**:
  - Rate each company on a scale of 1-100
  - Provide reasoning for each rating
  - Rank companies highest to lowest
- **Output**: Ranked list with ratings and reasons
- **Output File**: `output/decision.md`

## Task Flow

### Task 1: find_trending_companies_grok
```yaml
description: Find PUBLICLY TRADED companies with focus on fundamentals
agent: trending_company_finder_grok
output: TrendingCompanies (Pydantic model)
```

### Task 2: find_trending_companies_gemini
```yaml
description: Find PUBLICLY TRADED companies with focus on momentum
agent: trending_company_finder_gemini
output: TrendingCompanies (Pydantic model)
```

### Task 3: research_trending_companies
```yaml
description: Research ALL companies from BOTH lists
agent: financial_researcher
context:
  - find_trending_companies_grok
  - find_trending_companies_gemini
output: TrendingCompanyResearchList (Pydantic model)
```

### Task 4: rate_trending_companies
```yaml
description: Rate all researched companies 1-100
agent: stock_rater
context:
  - research_trending_companies
output: Markdown report
```

## Execution Flow

1. **Manager assigns Task 1** → Grok Finder searches for fundamental plays
   - Example output: NVDA, MSFT, GOOGL (strong fundamentals)

2. **Manager assigns Task 2** → Gemini Finder searches for momentum plays
   - Example output: SMCI, PLTR, SNOW (high momentum)

3. **Manager assigns Task 3** → Researcher receives BOTH lists
   - Combined list: NVDA, MSFT, GOOGL, SMCI, PLTR, SNOW
   - Removes duplicates if any overlap
   - Researches all 6 companies thoroughly

4. **Manager assigns Task 4** → Rater analyzes research
   - Rates each of the 6 companies
   - Example output:
     ```
     1. NVDA - 95/100 (Strong fundamentals + momentum)
     2. MSFT - 88/100 (Solid fundamentals)
     3. PLTR - 82/100 (High momentum)
     ...
     ```

## Key Benefits of This Design

### 1. Diverse Perspectives
- **Grok Finder**: Value/fundamental investing approach
- **Gemini Finder**: Growth/momentum investing approach
- **Result**: Covers both investment styles

### 2. Comprehensive Coverage
- Get 4-6 unique companies instead of just 2-3
- Mix of stable fundamentals and high-growth opportunities
- Reduces risk of missing good opportunities

### 3. Consistent Analysis
- **Single Researcher**: All companies analyzed with same methodology
- **Single Rater**: All companies rated with same criteria
- **Result**: Fair, comparable ratings

### 4. LLM Optimization
- **Gemini** for fast tasks (manager, momentum scanning)
- **Grok** for thorough tasks (fundamental analysis, research, rating)
- **Result**: Balance of speed and quality

## Pydantic Models

### TrendingCompany
```python
class TrendingCompany(BaseModel):
    name: str                      # Company name
    ticker: str                    # Stock ticker (REQUIRED - public only)
    price: float | None            # Current price (optional)
    projected_price: float | None  # 5-year projection (optional)
    reason: str                    # Why it's trending
```

### TrendingCompanies
```python
class TrendingCompanies(BaseModel):
    companies: list[TrendingCompany]  # List of 2-3 companies
```

### TrendingCompanyResearch
```python
class TrendingCompanyResearch(BaseModel):
    name: str                  # Company name
    market_position: str       # Market analysis
    future_outlook: str        # Growth prospects
    investment_potential: str  # Investment analysis
```

## Important Constraints

### Only Publicly Traded Companies
Both finders are instructed to:
- **ONLY** include companies with stock tickers
- **EXCLUDE** private companies (Perplexity, Cohere, etc.)
- **EXCLUDE** startups without tickers

This ensures all results are actually investable.

## Output Files

1. **trending_companies_grok.json**: Grok's fundamental picks
2. **trending_companies_gemini.json**: Gemini's momentum picks
3. **research_report.json**: Combined research on all companies
4. **decision.md**: Final ratings and recommendations

## Running the Crew

### Basic Run
```bash
cd ~/projects/agents/3_crew/my_stock_picker
crewai run
# Prompts for sector input (e.g., "technology")
```

### With Fallback
```bash
python run_with_diverse_opinions.py
# Automatically handles LLM failures with fallback chain
```

## LLM Fallback Strategy

The crew supports fallback configurations:

```python
llm_configs = [
    {  # Ideal: Diverse perspectives
        'manager': 'gemini-2.5-flash',
        'researcher': 'grok-4-fast',
        'analyst': 'gemini-2.5-flash',
        'rater': 'grok-4-fast'
    },
    {  # Fallback: All Gemini if Grok unavailable
        'manager': 'gemini-2.5-flash',
        'researcher': 'gemini-2.5-flash',
        'analyst': 'gemini-2.5-flash',
        'rater': 'gemini-2.5-flash'
    },
    {  # Final: All local Ollama
        'manager': 'ollama-gemma27b',
        'researcher': 'ollama-gemma27b',
        'analyst': 'ollama-gemma27b',
        'rater': 'ollama-gemma27b'
    }
]
```

## Comparison: Old vs New Design

| Aspect | Old Design | New Design |
|--------|-----------|------------|
| **Finders** | 1 agent | 2 agents (Grok + Gemini) |
| **Perspectives** | Single view | Dual perspectives (fundamental + momentum) |
| **Stock Coverage** | 2-3 companies | 4-6 companies |
| **Diversity** | Limited | High (two different "minds") |
| **Research** | Single list | Combined from two lists |
| **Rating** | Same | Same (consistent) |

## Why This Works Well

1. **Two Finders = Two Investment Styles**
   - Grok: Conservative, fundamental (like Warren Buffett)
   - Gemini: Aggressive, momentum (like growth traders)

2. **Single Research/Rating = Consistency**
   - All companies analyzed with same rigor
   - Fair comparison across different types of stocks

3. **Better Portfolio Balance**
   - Mix of stable (fundamental) and high-growth (momentum)
   - Reduces portfolio risk through diversification

4. **Leverages LLM Strengths**
   - Gemini: Fast scanning and coordination
   - Grok: Deep analysis and reasoning

# Running in Hierarchical Mode: my_stock_picker Project

## What is Hierarchical Process?

In CrewAI, there are two main execution modes:
- **Sequential**: Tasks run one after another in order
- **Hierarchical**: A manager agent coordinates and delegates tasks to worker agents

## Key Components for Hierarchical Mode

### 1. Process Type
```python
process=Process.hierarchical  # Instead of Process.sequential
```

### 2. Manager Configuration (Two Options)

#### Option A: manager_agent (Recommended - More Control)
Create a full Agent with specific configuration:
```python
manager = Agent(
    config=self.agents_config['manager'],
    llm=self.llm_gemini,
    allow_delegation=True,  # Required for hierarchical
    verbose=True
)

return Crew(
    agents=self.agents,
    tasks=self.tasks,
    process=Process.hierarchical,
    manager_agent=manager,  # Pass the full agent
    verbose=True,
)
```

**Advantages:**
- Full control over manager's role, goal, and backstory (defined in YAML)
- Can customize manager's behavior and personality
- Better for complex workflows where manager needs specific expertise

#### Option B: manager_llm (Simpler)
Just provide an LLM, CrewAI creates a generic manager:
```python
return Crew(
    agents=self.agents,
    tasks=self.tasks,
    process=Process.hierarchical,
    manager_llm=self.llm_gemini,  # Just pass the LLM
    verbose=True,
)
```

**Advantages:**
- Simpler code
- CrewAI handles manager creation automatically
- Good for straightforward delegation workflows

**Disadvantages:**
- Less control over manager behavior
- Generic manager without specific domain expertise

## my_stock_picker Project Structure

### Agents (from agents.yaml)
1. **trending_company_finder**: Finds 2-3 trending publicly traded companies
2. **financial_researcher**: Researches and analyzes the companies
3. **stock_rater**: Rates companies on investment potential (1-100 scale)
4. **manager**: Coordinates all agents and delegates tasks

### Tasks (from tasks.yaml)
1. **find_trending_companies**: Search for trending stocks
   - Output: JSON list with ticker, price, reason
   
2. **research_trending_companies**: Deep analysis of found companies
   - Uses context from task 1
   - Output: Detailed research report
   
3. **rate_trending_companies**: Rate investment potential
   - Uses context from task 2
   - Output: Ranked list with ratings (1-100)

### Workflow
```
Manager
  ├─> Delegates to trending_company_finder
  │     └─> Completes find_trending_companies task
  │
  ├─> Delegates to financial_researcher  
  │     └─> Completes research_trending_companies task
  │
  └─> Delegates to stock_rater
        └─> Completes rate_trending_companies task
```

## Implementation in my_stock_picker/src/my_stock_picker/crew.py

```python
@crew
def crew(self) -> Crew:
    """Creates the MyStockPicker crew"""

    # Create a manager agent with specific configuration
    manager = Agent(
        config=self.agents_config['manager'],
        llm=self.llm_grok,  # Using Grok for the manager
        allow_delegation=True,  # REQUIRED for hierarchical
        verbose=True
    )

    return Crew(
        agents=self.agents,  # Worker agents (auto-created by @agent decorator)
        tasks=self.tasks,    # Tasks (auto-created by @task decorator)
        process=Process.hierarchical,  # KEY: Hierarchical mode
        manager_agent=manager,  # Pass the configured manager
        verbose=True,
    )
```

## Key Differences: Sequential vs Hierarchical

| Aspect | Sequential | Hierarchical |
|--------|-----------|-------------|
| **Execution** | Tasks run in order | Manager delegates tasks |
| **Coordination** | Implicit (task order) | Explicit (manager decides) |
| **Agent Assignment** | Tasks assigned to specific agents in YAML | Manager chooses which agent to assign |
| **Complexity** | Simpler | More complex |
| **Use Case** | Linear workflows | Complex coordination, parallel work |
| **Manager** | Not needed | Required (manager_agent or manager_llm) |

## Code Quality Notes

### Current Implementation
The current code is well-structured but could be improved:

1. **Manager LLM Choice**: Uses `llm_grok` for manager
   - This is reasonable for complex decision-making
   - Could also use `llm_gemini` for faster coordination

2. **Pydantic Models**: Good use of structured outputs
   - `TrendingCompanies` for task 1
   - `TrendingCompanyResearchList` for task 2
   - No Pydantic for task 3 (just markdown output)

3. **Tool Usage**: SerperDevTool assigned to multiple agents
   - Good: Agents that need web search have the tool
   - Note: Manager doesn't need tools (just delegates)

### Potential Improvements

1. **Add manager_llm fallback**:
```python
# If manager agent creation fails, fallback to manager_llm
try:
    manager = Agent(...)
    return Crew(manager_agent=manager, ...)
except Exception:
    return Crew(manager_llm=self.llm_gemini, ...)
```

2. **Add memory to agents** for better context retention:
```python
@agent
def researcher(self) -> Agent:
    return Agent(
        config=self.agents_config['researcher'],
        llm=self.llm_grok,
        memory=True,  # Remember previous interactions
        verbose=True,
        tools=[SerperDevTool()]
    )
```

3. **Add validation to Pydantic models**:
```python
from pydantic import field_validator

class TrendingCompany(BaseModel):
    ticker: str
    
    @field_validator('ticker')
    def validate_ticker(cls, v):
        if not v or len(v) > 5:
            raise ValueError('Invalid ticker symbol')
        return v.upper()
```

## Running the Project

From `3_crew/my_stock_picker` directory:
```bash
crewai run
```

The crew will prompt for `sector` input (e.g., "technology", "healthcare").

# LLM Fallback Strategy

## The Problem

When running crews in production, you may encounter:
- **Rate limits**: "Too many requests" from Gemini/Grok
- **Service outages**: API temporarily unavailable
- **Quota exhaustion**: Daily/monthly limits exceeded

## The Solution: Fallback Chain

Implement a fallback chain that tries multiple LLMs in order:

```
Gemini (fast, cheap) → Grok (capable, moderate) → Ollama (local, always available)
```

## Implementation

### 1. Update Crew __init__ to Accept Override

```python
def __init__(self, manager_llm_override=None):
    """
    Args:
        manager_llm_override: Override which LLM the manager uses
                             Options: 'gemini-2.5-flash', 'grok-4-fast', 'ollama-gemma27b'
    """
    llms = build_llms(['grok-4-fast', 'gemini-2.5-flash', 'ollama-gemma27b'])
    self.llm_grok = llms['grok-4-fast']
    self.llm_gemini = llms['gemini-2.5-flash']
    self.llm_ollama = llms['ollama-gemma27b']
    
    # Support dynamic manager LLM selection
    if manager_llm_override:
        llm_map = {
            'gemini-2.5-flash': self.llm_gemini,
            'grok-4-fast': self.llm_grok,
            'ollama-gemma27b': self.llm_ollama
        }
        self.manager_llm = llm_map.get(manager_llm_override, self.llm_gemini)
    else:
        self.manager_llm = self.llm_gemini  # Default
```

### 2. Use manager_llm in Crew Definition

```python
@crew
def crew(self) -> Crew:
    manager = Agent(
        config=self.agents_config['manager'],
        llm=self.manager_llm,  # Dynamic LLM based on override
        allow_delegation=True,
        verbose=True
    )
    
    return Crew(
        agents=self.agents,
        tasks=self.tasks,
        process=Process.hierarchical,
        manager_agent=manager,
        verbose=True,
    )
```

### 3. Create Fallback Runner

File: `3_crew/crew_runner_with_fallback.py`

```python
def run_crew_with_fallback(crew_class, manager_llms, inputs=None, max_retries=3):
    """
    Run crew with automatic fallback if LLM fails.
    
    Args:
        crew_class: Crew class to instantiate
        manager_llms: List of LLM names to try ['gemini-2.5-flash', 'grok-4-fast', ...]
        inputs: Input dict for crew
        max_retries: Max retries per LLM
    
    Returns:
        CrewOutput if successful
    """
    for llm_name in manager_llms:
        for attempt in range(max_retries):
            try:
                crew_instance = crew_class(manager_llm_override=llm_name)
                result = crew_instance.crew().kickoff(inputs=inputs)
                return result  # Success!
            
            except Exception as e:
                # Check if rate limit error
                if 'rate limit' in str(e).lower() or '429' in str(e):
                    if attempt < max_retries - 1:
                        wait_time = 2 ** attempt  # Exponential backoff
                        time.sleep(wait_time)
                        continue
                    else:
                        break  # Move to next LLM
                else:
                    break  # Non-rate-limit error, move to next LLM
    
    raise Exception("All LLMs failed")
```

### 4. Usage Example

```python
from my_stock_picker.crew import MyStockPicker
from crew_runner_with_fallback import run_crew_with_fallback

result = run_crew_with_fallback(
    crew_class=MyStockPicker,
    manager_llms=['gemini-2.5-flash', 'grok-4-fast', 'ollama-gemma27b'],
    inputs={'sector': 'technology'},
    max_retries=2
)
```

## Execution Flow

```
1. Try Gemini with input
   └─ Rate limit → Retry (wait 1s)
      └─ Rate limit again → Move to next LLM

2. Try Grok with input
   └─ Success! → Return result

(If Grok also failed, would try Ollama)
```

## Best Practices

### 1. Order Your Fallback Chain Strategically

```python
# Fast → Capable → Always Available
['gemini-2.5-flash', 'grok-4-fast', 'ollama-gemma27b']
```

**Reasoning:**
- **Gemini**: Fastest and cheapest, try first
- **Grok**: More capable reasoning, good fallback
- **Ollama**: Local model, always works (no rate limits)

### 2. Implement Exponential Backoff

```python
wait_time = 2 ** attempt  # 1s, 2s, 4s, 8s...
```

Avoids hammering the API immediately after failure.

### 3. Detect Different Error Types

```python
error_msg = str(e).lower()

if any(keyword in error_msg for keyword in 
       ['rate limit', 'quota', 'overloaded', '429', 'too many requests']):
    # Retry or fallback
elif 'authentication' in error_msg or '401' in error_msg:
    # Don't retry, fix API key
elif 'timeout' in error_msg:
    # Network issue, maybe retry with longer timeout
```

### 4. Log Everything

```python
import logging

logger.info(f"Attempting crew with {llm_name}")
logger.warning(f"Rate limit on {llm_name}, attempt {attempt}")
logger.error(f"All LLMs failed. Last error: {e}")
```

Helps debug which LLM failed and why.

### 5. Consider Cost vs Reliability

| LLM | Speed | Cost | Reliability | When to Use |
|-----|-------|------|-------------|-------------|
| Gemini Flash | ⚡⚡⚡ | 💰 | 🔄 (rate limits) | First choice, development |
| Grok | ⚡⚡ | 💰💰 | 🔄🔄 (fewer limits) | Production fallback |
| Ollama | ⚡ | Free | ✅ Always works | Final fallback, local dev |

## Alternative: Worker Agent Fallbacks

You could also apply fallbacks to individual worker agents:

```python
@agent
def researcher(self) -> Agent:
    # Try Grok first for research, fallback to Gemini
    llm = self.llm_grok if not hasattr(self, '_research_failed') else self.llm_gemini
    
    return Agent(
        config=self.agents_config['researcher'],
        llm=llm,
        verbose=True,
        tools=[SerperDevTool()]
    )
```

But manager fallback is usually sufficient since the manager coordinates everything.

## Files Created

1. **3_crew/crew_runner_with_fallback.py**: Reusable fallback runner
2. **my_stock_picker/run_with_fallback.py**: Example usage script
3. **my_stock_picker/src/my_stock_picker/crew.py**: Updated with manager_llm_override support

# How the last my_stock_picker works